# 04. Tools and Structured Output

Learn how to build custom tools with the `@tool` decorator in LangChain v1 and how to receive structured responses with `with_structured_output()`.


## Learning Objectives

- Build tools with the `@tool` decorator and inspect their schemas
- Define complex input schemas with Pydantic models
- Connect tools to `create_agent()` and build an agent
- Access runtime context from inside a tool with `ToolRuntime`
- Configure structured output with `with_structured_output()`
- Understand the difference between `ToolStrategy` and `ProviderStrategy`


## 4.1 Environment Setup

Load API keys and initialize an OpenAI model.


In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

# Initialize the model with OpenAI
model = ChatOpenAI(
    model="gpt-4.1",
)

print("Model initialized:", model.model_name)

In [ ]:
# Optional observability setup: LangSmith or Langfuse
# Set the keys in .env, or uncomment the lines below to enter them manually.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: automatically enabled when LANGSMITH_TRACING=true
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: pass config={"callbacks": [langfuse_handler]} to invoke/stream
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 4.2 The Basics of the `@tool` Decorator

When you add `@tool` to a function, it becomes a tool that an agent can use.  
LangChain automatically parses the function name, docstring, and type hints to build the tool schema.

```python
from langchain.tools import tool

@tool
def my_tool(param: str) -> str:
    """Tool description for the LLM."""
    return result
```


In [ ]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Look up the current weather for a city."""
    weather_data = {
        "Seoul": "맑음, 15\u00b0C",
        "Tokyo": "흐림, 12\u00b0C",
        "New York": "비, 8\u00b0C",
    }
    return weather_data.get(city, f"Weather data is not available for: {city}")

# Inspect the tool schema
print("Tool name:", get_weather.name)
print("Tool description:", get_weather.description)
print("Input schema:", get_weather.args_schema.model_json_schema())

## 4.3 Complex Schemas with Pydantic

If you need a richer input structure, define the schema with a Pydantic `BaseModel`.  
When you pass it as `@tool(args_schema=MySchema)`, the LLM can understand the exact parameter structure.

- `Field(description=...)`: passes a field description to the LLM
- `Field(default=...)`: defines a default value


In [ ]:
from pydantic import BaseModel, Field

class SearchQuery(BaseModel):
    """Search parameters for a database query."""
    query: str = Field(description="Search query string")
    max_results: int = Field(default=5, description="Maximum number of results to return")
    category: str = Field(default="all", description="Search category: all, tech, science, news")

@tool(args_schema=SearchQuery)
def search_database(query: str, max_results: int = 5, category: str = "all") -> str:
    """Search the database with advanced filtering options."""
    return f"'{category}' 카테고리에서 '{query}'에 대한 {max_results}개의 결과를 찾았습니다"

print("Complex schema:", search_database.args_schema.model_json_schema())

## 4.4 Connecting Tools to an Agent

When you pass a list of tools into `create_agent()`, the agent can automatically choose and execute the right tool for the situation.

```python
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[tool1, tool2],
    system_prompt="...",
)
```

> **Note:** In LangChain v1, `create_react_agent` was removed. Always use `create_agent`.


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[get_weather, search_database],
    system_prompt="You are an assistant that can use weather and search tools.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather like in Seoul?"}]},
    config=lf_config,
)
print("Agent response:", result["messages"][-1].content)

## 4.5 ToolRuntime

`ToolRuntime` lets a tool function access the current conversation state at runtime.  
This makes it possible to build tools that use message history, settings, or other runtime context.

```python
@tool
def my_tool(runtime: ToolRuntime) -> str:
    messages = runtime.state["messages"]
    # ...
```


In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def get_user_info(runtime: ToolRuntime) -> str:
    """Get information about the current conversation."""
    messages = runtime.state["messages"]
    return f"There are {len(messages)} messages in the current conversation."

agent_with_runtime = create_agent(
    model=model,
    tools=[get_user_info],
    system_prompt="You can use the get_user_info tool to inspect the conversation.",
)

result = agent_with_runtime.invoke(
    {"messages": [{"role": "user", "content": "How many messages are in our conversation?"}]},
    config=lf_config,
)
print("Response:", result["messages"][-1].content)

## 4.6 Structured Output

With `with_structured_output()`, you can receive the model's response directly as a Pydantic model or dataclass.  
This pattern is used directly on the model, not through an agent.

```python
structured_model = model.with_structured_output(MySchema)
result = structured_model.invoke("...")
# result is an instance of MySchema
```


In [ ]:
# Method 1: with_structured_output() -- use the model directly
from pydantic import BaseModel

class MovieReview(BaseModel):
    """Structured movie review."""
    title: str
    rating: float
    summary: str
    recommended: bool

structured_model = model.with_structured_output(MovieReview)

review = structured_model.invoke("Review Christopher Nolan's film 'Inception'.", config=lf_config)
print(f"Title: {review.title}")
print(f"Rating: {review.rating}")
print(f"Summary: {review.summary}")
print(f"Recommended: {'yes' if review.recommended else 'no'}")

## 4.7 ToolStrategy vs ProviderStrategy

There are two main strategies for structured output inside an agent:

| Strategy | Description | Advantage |
|------|------|------|
| `ToolStrategy` | Uses the tool-calling mechanism to produce structured output | Works with every model and is stable |
| `ProviderStrategy` | Uses the provider's native structured-output feature | Faster and more accurate when the model supports it |

Use the `response_format` parameter to structure the agent's final response.


In [ ]:
from langchain.agents.structured_output import ToolStrategy
from dataclasses import dataclass

@dataclass
class CalculationResult:
    """Calculation result."""
    expression: str
    result: float
    explanation: str

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        result = eval(expression)
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error: {e}"

agent_structured = create_agent(
    model=model,
    tools=[calculate],
    system_prompt="You are a math assistant. Always use the calculate tool.",
    response_format=ToolStrategy(CalculationResult),
)

result = agent_structured.invoke(
    {"messages": [{"role": "user", "content": "What is 2 to the 10th power?"}]},
    config=lf_config,
)
print("구조화된 Response:", result.get("structured_response"))

## 4.8 Summary

Here is a summary of the main ideas in this notebook.

| Item | Description |
|------|------|
| `@tool` decorator | Converts a function into an agent tool |
| `args_schema` | Defines a complex input schema with Pydantic |
| `create_agent()` | Connects the model and tools to create an agent |
| `ToolRuntime` | Gives a tool access to runtime state such as conversation history |
| `with_structured_output()` | Structures model output as a Pydantic model or dataclass |
| `ToolStrategy` | Structured agent output through tool calling |
| `ProviderStrategy` | Provider-native structured output |

### Next Steps
→ **[05_memory_and_streaming.ipynb](./05_memory_and_streaming.ipynb)**: Learn about memory and streaming.
